# Semi-Analytic Population Model

This notebook introduces the core semi-analytic SMBHB population model used in `fastropop`. It shows the main population-level quantities before any Monte Carlo sampling is performed.

## Imports and Setup

We import the package, choose one concrete parameter set, and instantiate `SemiAnalyticPopulation`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import jax.numpy as jnp

import fastropop
from fastropop import SemiAnalyticPopulation


In [ ]:
population_params = {
    "n0": 10**-90.4153,
    "alphaM": -1.3800,
    "Mstar": 10**8.8272 * fastropop.MsunMKS,
    "betaz": -0.1711,
    "z0": 4.70,
}

sampling_grids = {
    "Mgrid": jnp.geomspace(fastropop.Mmin / fastropop.kg, fastropop.Mmax / fastropop.kg, 300),
    "zgrid": jnp.linspace(fastropop.zmin, fastropop.zmax, 300),
    "fgrid": jnp.geomspace(fastropop.fminNG15 * fastropop.s, fastropop.fmaxNG15 * fastropop.s, 300),
}

pop = SemiAnalyticPopulation(
    population_params=population_params,
    sampling_grids=sampling_grids,
)


## Merger-Rate Density as a Function of Mass

The first plot shows `d2ndzdM` at fixed redshift. This is the mass-dependent part of the semi-analytic population model after converting from `d/dlog10M` to `d/dM`.

In [ ]:
M_vals = np.logspace(6, 11, 500)
y_vals = np.array([pop.d2ndzdM(1.0, M * fastropop.MsunMKS) for M in M_vals])

plt.figure(figsize=(7, 5))
plt.loglog(M_vals, y_vals, color="C1")
plt.xlabel(r"$M\ [M_\odot]$")
plt.ylabel(r"$d^2 n / (dz\, dM)$")
plt.title("Mass Dependence at z = 1")
plt.grid(True, which="both", ls="--", alpha=0.5)
plt.show()


## Integrated Mass Function

Next we integrate over redshift to obtain `dndlog10M`, which is often easier to interpret as a mass function.

In [ ]:
M_vals = np.logspace(6, 10, 100) * fastropop.MsunMKS
dndlogM_vals = np.array([pop.dndlog10M(M) for M in M_vals])

plt.figure(figsize=(7, 5))
plt.loglog(M_vals / fastropop.MsunMKS, dndlogM_vals)
plt.xlabel(r"$M / M_\odot$")
plt.ylabel(r"$dn / d\log_{10} M\ [\mathrm{Mpc^{-3}}]$")
plt.ylim(1e-9, 1e3)
plt.grid(True, which="both", ls="--", alpha=0.5)
plt.show()


## Ensemble Characteristic Strain

Finally, we evaluate the smooth ensemble characteristic-strain spectrum `h_c(f)` implied by the model. This is the reference curve before any Poisson fluctuations are added.

In [ ]:
freqs = 10 ** np.arange(-9.0, -7.0, 0.2)
hc_values = np.array([np.sqrt(pop.hc2(float(f))) for f in freqs])

plt.figure(figsize=(7, 5))
plt.semilogx(freqs, np.log10(hc_values), marker="o", color="C0", label="Numerical")
plt.semilogx(
    freqs,
    -(np.log10(freqs[0] ** (-2 / 3)) - np.log10(hc_values[0])) + np.log10(freqs ** (-2 / 3)),
    "--",
    color="C1",
    label=r"$f^{-2/3}$",
)
plt.xlabel(r"$f\ [\mathrm{Hz}]$")
plt.ylabel(r"$\log_{10}(h_c)$")
plt.title("Ensemble Characteristic Strain")
plt.grid(True, which="both", ls="--", alpha=0.5)
plt.legend()
plt.show()
